In [26]:
# imports and setup

import os
import base64
import sys
from dotenv import load_dotenv
from typing import Any, Dict, List, Optional, Tuple
from PIL import Image   # noqa: F401
from pydantic import BaseModel as PydanticBaseModel
from pydantic import Field
from pydantic_settings import BaseSettings
import oracledb
import io
import cv2
import fitz
import numpy as np
import boto3
from botocore.config import Config
import json
import re
import time

# Add project root to Python path so we can import from app module
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)
# Now we can import from app (after adding to sys.path)
from app.utils.logger import get_logger
# Load environment variables from the project root directory
env_path = os.path.join(project_root, '.env')
load_dotenv(env_path)

logger = get_logger(name=__name__)

In [27]:

class AppConstants:
    """Constantes globais para a aplicação."""

    BEDROCK_DEFAULT_MODEL_ID = "anthropic.claude-3-7-sonnet-20240729-v1:0"
    # DEFAULT_PROMPT_EXTRACAO_PATH = "prompt_extracao_laudo.txt"
    # DEFAULT_PROMPT_RESUMO_PATH = "prompt_resumo.txt"
    # S3_BUCKET_NAME = "agente-ai-laudos"
    # S3_RESULTS_PREFIX = "resultados"
    # S3_DEBUG_PREFIX = "debug"
    MAX_RETRIES = 5
    INITIAL_BACKOFF_SECONDS = 2


In [28]:
class Settings(BaseSettings):
    """Carrega e valida as configurações a partir de variáveis de ambiente."""

    ORACLE_USER: str
    ORACLE_PASSWORD: str
    ORACLE_DSN: str
    ORACLE_INSTANT_CLIENT_PATH: Optional[str] = Field(
        None, alias="oracle_instant_client_path"
    )
    AWS_ACCESS_KEY_ID: str
    AWS_SECRET_ACCESS_KEY: str
    AWS_BEDROCK_REGION: str
    BEDROCK_MODEL_ID: str = AppConstants.BEDROCK_DEFAULT_MODEL_ID
    MARIADB_USER: str
    MARIADB_PASSWORD: str
    MARIADB_HOST: str
    MARIADB_PORT: int = 3306
    MARIADB_DATABASE: str
    API_BASE_URL: Optional[str] = Field(None, alias="api_base_url")
    API_USERNAME: Optional[str] = Field(None, alias="username")
    API_PASSWORD: Optional[str] = Field(None, alias="password")

    class Config:
        env_file = ".env"
        env_file_encoding = "utf-8"


In [29]:
def criar_boto3_client(
    service_name: str, settings: Settings, config: Optional[Config] = None
) -> boto3.client:
    try:
        logger.info(
            f"Criando cliente {service_name.upper()} para a região: {settings.AWS_BEDROCK_REGION}..."
        )
        client = boto3.client(
            service_name,
            region_name=settings.AWS_BEDROCK_REGION,
            aws_access_key_id=settings.AWS_ACCESS_KEY_ID,
            aws_secret_access_key=settings.AWS_SECRET_ACCESS_KEY,
            config=config,
        )
        logger.info(f"Cliente {service_name.upper()} criado com sucesso.")
        return client
    except Exception as e:
        logger.critical(f"Não foi possível criar o cliente {service_name.upper()}: {e}")
        raise


In [30]:
# connection to the database

class OracleService:
    def __init__(self, settings: Settings):
        self.config = {
            "user": settings.ORACLE_USER,
            "password": settings.ORACLE_PASSWORD,
            "dsn": settings.ORACLE_DSN,
        }
        self.connection = None
        if settings.ORACLE_INSTANT_CLIENT_PATH and os.path.isdir(
            settings.ORACLE_INSTANT_CLIENT_PATH
        ):
            logger.info(
                f"Inicializando Oracle Client de: {settings.ORACLE_INSTANT_CLIENT_PATH}"
            )
            oracledb.init_oracle_client(lib_dir=settings.ORACLE_INSTANT_CLIENT_PATH)

    def __enter__(self):
        """Establishes the database connection when entering the 'with' block."""
        try:
            self.connection = oracledb.connect(**self.config)
            logger.info(f"✅ Connection established to {self.config['dsn']}")
            return self
        except Exception as e:
            logger.error(f"❌ Error connecting to Oracle: {e}")
            raise

    def __exit__(self, exc_type, exc_val, exc_tb):
        """Closes the database connection when exiting the 'with' block."""
        if self.connection:
            try:
                self.connection.close()
                logger.info("✅ Oracle connection closed.")
            except Exception as e:
                logger.error(f"❌ Error closing Oracle connection: {e}")
        # If an exception occurred within the 'with' block, it will be re-raised.

    def execute_query(self, query: str, fetch_limit: int = None) -> Optional[List[Dict[str, Any]]]:
        """
        Execute a query using the existing Oracle connection.
        
        Args:
            query: SQL query to execute
            fetch_limit: Maximum number of rows to fetch (None for all)
        
        Returns:
            List of dictionaries containing query results, or None if error
        """
        if not self.connection:
            logger.error("❌ Cannot execute query: Not connected. Use within a 'with' block.")
            return None
            
        try:
            with self.connection.cursor() as cursor:
                logger.info("Executing query...")
                cursor.execute(query)
                
                columns = [desc[0] for desc in cursor.description]
                
                if fetch_limit:
                    rows = cursor.fetchmany(fetch_limit)
                else:
                    rows = cursor.fetchall()
                
                results = []
                for row in rows:
                    row_dict = {}
                    for i, col_val in enumerate(row):
                        col_name = columns[i]
                        if isinstance(col_val, oracledb.LOB):
                            row_dict[col_name] = col_val.read()
                        else:
                            row_dict[col_name] = col_val
                    results.append(row_dict)
                
                logger.info(f"✅ Fetched {len(results)} rows.")
                return results
                    
        except Exception as e:
            logger.error(f"Error executing query: {e}")
            return None

In [31]:
# Load Query from External File

def load_query_from_file(file_path: str) -> str:
    """
    Load SQL query from a text file
    
    Args:
        file_path: Path to the query file
    
    Returns:
        Query string
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
            
        # Extract just the query part (remove variable assignment)
        if '"""' in content:
            # Find the query between triple quotes
            start = content.find('"""') + 3
            end = content.rfind('"""')
            query = content[start:end].strip()
        else:
            # If no triple quotes, assume entire file is the query
            query = content.strip()
            
        return query
        
    except Exception as e:
        logger.error(f"❌ Error loading query from file: {e}")
        return None

In [32]:
# Load the carteirinha query from file
query_file_path = os.path.join(os.getcwd(), 'query_carteirinha.txt')
convenio_carteirinha_query = load_query_from_file(query_file_path)

if convenio_carteirinha_query:
    logger.info("✅ Carteirinha query loaded successfully from file!")
    logger.info(f"📄 Query length: {len(convenio_carteirinha_query)} characters")
    logger.info(f"📝 First 100 characters: {convenio_carteirinha_query[:100]}...")
else:
    logger.error("❌ Failed to load query from file")

{"timestamp": "2025-08-06T12:50:18", "level": "INFO", "name": "__main__", "message": "✅ Carteirinha query loaded successfully from file!", "filename": "600200798.py", "lineno": 6}
{"timestamp": "2025-08-06T12:50:18", "level": "INFO", "name": "__main__", "message": "📄 Query length: 2960 characters", "filename": "600200798.py", "lineno": 7}
{"timestamp": "2025-08-06T12:50:18", "level": "INFO", "name": "__main__", "message": "📝 First 100 characters: SELECT\n    dac.cd_aviso_cirurgia,\n    dac.CD_DOCUMENTO_ANEXO_CIRURGICO,\n    g.cd_guia,\n    g.tp_guia...", "filename": "600200798.py", "lineno": 8}
{"timestamp": "2025-08-06T12:50:18", "level": "INFO", "name": "__main__", "message": "📄 Query length: 2960 characters", "filename": "600200798.py", "lineno": 7}
{"timestamp": "2025-08-06T12:50:18", "level": "INFO", "name": "__main__", "message": "📝 First 100 characters: SELECT\n    dac.cd_aviso_cirurgia,\n    dac.CD_DOCUMENTO_ANEXO_CIRURGICO,\n    g.cd_guia,\n    g.tp_guia...", "filename": "6002

In [33]:
# Test the Carteirinha Query using the OracleService context manager

logger.info("🚀 Testing carteirinha query with context manager...")
logger.info("=" * 50)

results = None
try:
    settings = Settings()
    
    # Use the service as a context manager
    with OracleService(settings) as oracle_service:
        if convenio_carteirinha_query:
            results = oracle_service.execute_query(convenio_carteirinha_query, fetch_limit=5)

            if results:
                logger.info(f"\n✅ Query executed successfully! Found {len(results)} sample records")
                logger.info("\n📋 Sample Results:")
                logger.info("-" * 50)
                
                for i, record in enumerate(results, 1):
                    logger.info(f"\nRecord {i}:")
                    for key, value in record.items():
                        if key == 'LO_DOCUMENTO_ANEXO_CIRURGICO':
                            logger.info(f"  {key}: {'BLOB data present' if value else 'No BLOB data'}")
                        else:
                            logger.info(f"  {key}: {value}")
                
                logger.info(f"\n📊 To get all records, run the query without fetch_limit.")
                
            else:
                logger.error("❌ No results returned or query failed inside 'with' block")
        else:
            logger.error("❌ Query not loaded - cannot execute")

except Exception as e:
    logger.error(f"❌ An error occurred outside the 'with' block: {e}")

logger.info("=" * 50)

{"timestamp": "2025-08-06T12:50:18", "level": "INFO", "name": "__main__", "message": "🚀 Testing carteirinha query with context manager...", "filename": "1278188058.py", "lineno": 3}
{"timestamp": "2025-08-06T12:50:18", "level": "INFO", "name": "__main__", "message": "==================================================", "filename": "1278188058.py", "lineno": 4}
{"timestamp": "2025-08-06T12:50:18", "level": "INFO", "name": "__main__", "message": "==================================================", "filename": "1278188058.py", "lineno": 4}
{"timestamp": "2025-08-06T12:50:19", "level": "INFO", "name": "__main__", "message": "✅ Connection established to srvhmddb007-otk6s-scan.sbntdb.vcnprod.oraclevcn.com:1521/PRDREPT_OCI", "filename": "1417701281.py", "lineno": 23}
{"timestamp": "2025-08-06T12:50:19", "level": "INFO", "name": "__main__", "message": "Executing query...", "filename": "1417701281.py", "lineno": 56}
{"timestamp": "2025-08-06T12:50:19", "level": "INFO", "name": "__main__", "mes

In [34]:
# image utils
def converter_blob_para_imagens(blob: bytes, extensao: str) -> List[Image.Image]:
    imagens = []
    ext = extensao.lower().strip(".") if extensao else ""
    try:
        if ext == "pdf":
            with fitz.open(stream=blob, filetype="pdf") as pdf_doc:
                logger.info(f"Processando PDF com {len(pdf_doc)} página(s)...")
                for pagina in pdf_doc:
                    pix = pagina.get_pixmap(matrix=fitz.Matrix(3.0, 3.0), alpha=False)
                    imagens.append(Image.open(io.BytesIO(pix.tobytes("png"))))
        elif ext in ["jpg", "jpeg", "png", "bmp"]:
            imagens.append(Image.open(io.BytesIO(blob)))
        else:
            logger.warning(f"Formato de arquivo não suportado: '{ext}'.")
    except Exception as e:
        logger.error(f"Erro ao converter BLOB para imagem (ext: .{ext}): {e}")
    return imagens


def aplicar_clahe(imagem: Image.Image) -> Image.Image:
    try:
        imagem_cv = cv2.cvtColor(np.array(imagem), cv2.COLOR_RGB2BGR)
        imagem_cinza = cv2.cvtColor(imagem_cv, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        return Image.fromarray(clahe.apply(imagem_cinza))
    except Exception:
        return imagem


def imagem_para_base64(imagem: Image.Image) -> str:
    TARGET_BYTES = 3.5 * 1024 * 1024
    if imagem.mode in ("RGBA", "P"):
        imagem = imagem.convert("RGB")
    for quality in range(95, 15, -10):
        buffer = io.BytesIO()
        imagem.save(buffer, format="JPEG", quality=quality)
        if buffer.tell() <= TARGET_BYTES:
            if quality < 95:
                logger.warning(
                    f"Imagem comprimida (qualidade {quality}%) para caber no limite."
                )
            return base64.b64encode(buffer.getvalue()).decode("utf-8")
    raise ValueError(
        f"Não foi possível reduzir a imagem abaixo de {TARGET_BYTES / (1024 * 1024):.1f}MB."
    )


In [35]:
# extract blob data from the results
blobs = []
if results:
    for record in results:
        blob_data = record.get('LO_DOCUMENTO_ANEXO_CIRURGICO')
        if blob_data and isinstance(blob_data, bytes):
            blobs.append(blob_data)
            logger.info(f"✅ Extracted BLOB of size {len(blob_data)} bytes")
        else:
            logger.warning("No BLOB data found in record or data is not bytes")

if blobs:
    logger.info(f"Successfully extracted {len(blobs)} BLOBs into a list.")
else:
    logger.error("Could not extract any BLOBs from the results.")

{"timestamp": "2025-08-06T12:50:24", "level": "INFO", "name": "__main__", "message": "✅ Extracted BLOB of size 160226 bytes", "filename": "395405530.py", "lineno": 8}
{"timestamp": "2025-08-06T12:50:24", "level": "INFO", "name": "__main__", "message": "✅ Extracted BLOB of size 153988 bytes", "filename": "395405530.py", "lineno": 8}
{"timestamp": "2025-08-06T12:50:24", "level": "INFO", "name": "__main__", "message": "✅ Extracted BLOB of size 111662 bytes", "filename": "395405530.py", "lineno": 8}
{"timestamp": "2025-08-06T12:50:24", "level": "INFO", "name": "__main__", "message": "✅ Extracted BLOB of size 616508 bytes", "filename": "395405530.py", "lineno": 8}
{"timestamp": "2025-08-06T12:50:24", "level": "INFO", "name": "__main__", "message": "✅ Extracted BLOB of size 506822 bytes", "filename": "395405530.py", "lineno": 8}
{"timestamp": "2025-08-06T12:50:24", "level": "INFO", "name": "__main__", "message": "Successfully extracted 5 BLOBs into a list.", "filename": "395405530.py", "line

In [36]:
# take the list of blobs and convert them to images. them, save them as base64 strings
base64_images = []
if blobs:
    for i, blob in enumerate(blobs):
        logger.info(f"Converting BLOB {i+1}/{len(blobs)} to images...")
        imagens = converter_blob_para_imagens(blob, "pdf")  # Assuming PDF for this example
        # save the images on ../documents/carteirinhas_images/
        if not os.path.exists("../documents/carteirinhas_images/"):
            os.makedirs("../documents/carteirinhas_images/")
        for j, img in enumerate(imagens):
            img_path = f"../documents/carteirinhas_images/blob_{i+1}_image_{j+1}.png"
            img.save(img_path)
            logger.info(f"Saved image {j+1} from BLOB {i+1} to {img_path}")
        logger.info(f"Extracted {len(imagens)} images from BLOB {i+1}.")

        if imagens:
            for j, img in enumerate(imagens):
                logger.info(f"Processing image {j+1} from BLOB {i+1}...")
                img_clahe = aplicar_clahe(img)
                base64_str = imagem_para_base64(img_clahe)
                base64_images.append(base64_str)
                logger.info(f"✅ Converted image {j+1} to base64 string.")
        else:
            logger.warning(f"No images extracted from BLOB {i+1}.")

{"timestamp": "2025-08-06T12:50:24", "level": "INFO", "name": "__main__", "message": "Converting BLOB 1/5 to images...", "filename": "4197694994.py", "lineno": 5}
{"timestamp": "2025-08-06T12:50:24", "level": "INFO", "name": "__main__", "message": "Processando PDF com 1 página(s)...", "filename": "2299205648.py", "lineno": 8}
{"timestamp": "2025-08-06T12:50:24", "level": "INFO", "name": "__main__", "message": "Processando PDF com 1 página(s)...", "filename": "2299205648.py", "lineno": 8}
{"timestamp": "2025-08-06T12:50:24", "level": "INFO", "name": "__main__", "message": "Saved image 1 from BLOB 1 to ../documents/carteirinhas_images/blob_1_image_1.png", "filename": "4197694994.py", "lineno": 13}
{"timestamp": "2025-08-06T12:50:24", "level": "INFO", "name": "__main__", "message": "Extracted 1 images from BLOB 1.", "filename": "4197694994.py", "lineno": 14}
{"timestamp": "2025-08-06T12:50:24", "level": "INFO", "name": "__main__", "message": "Processing image 1 from BLOB 1...", "filename"

In [37]:
base64_images_count = len(base64_images)
if base64_images_count > 0:
    logger.info(f"Successfully converted {base64_images_count} images to base64 strings.")
else:
    logger.warning("No images were converted to base64 strings. Check the BLOB data.")

{"timestamp": "2025-08-06T12:50:25", "level": "INFO", "name": "__main__", "message": "Successfully converted 5 images to base64 strings.", "filename": "1948280603.py", "lineno": 3}


In [38]:
class CarteirinhaExtraida(PydanticBaseModel):
    """Define a estrutura dos dados extraídos da carteirinha."""
    convenio: Optional[str] = Field(None, description="Nome do convênio de saúde.")
    plano: Optional[str] = Field(None, description="Nome do plano de saúde.")
    nome_pessoa: Optional[str] = Field(None, description="Nome completo do titular ou beneficiário.")
    numero_carteirinha: Optional[str] = Field(None, description="O número de identificação da carteirinha.")

In [39]:
class LLMService:
    """Encapsula a lógica de chamada ao modelo de linguagem (Bedrock)."""

    def __init__(self, bedrock_client: boto3.client, model_id: str):
        self.bedrock_client = bedrock_client
        self.model_id = model_id

    def _invocar_llm(self, corpo_requisicao: Dict, context_log: str = "") -> Dict:
        resultado = {
            "success": False,
            "dados": None,
            "raw_response": "",
            "error": None,
            "input_tokens": 0,
            "output_tokens": 0,
        }
        retries = 0
        while retries < AppConstants.MAX_RETRIES:
            try:
                response = self.bedrock_client.invoke_model(
                    body=json.dumps(corpo_requisicao), modelId=self.model_id
                )
                response_body = json.loads(response.get("body").read())
                usage = response_body.get("usage", {})
                raw_text = response_body.get("content", [{}])[0].get("text", "")
                resultado.update(
                    {
                        "input_tokens": usage.get("input_tokens", 0),
                        "output_tokens": usage.get("output_tokens", 0),
                        "raw_response": raw_text,
                    }
                )
                json_str = re.search(r"\{.*\}", raw_text, re.DOTALL)
                if not json_str:
                    raise ValueError(
                        "Nenhum JSON válido encontrado na resposta do LLM."
                    )
                resultado["dados"] = json.loads(json_str.group(0))
                resultado["success"] = True
                return resultado
            except self.bedrock_client.exceptions.ThrottlingException as e:
                retries += 1
                wait_time = AppConstants.INITIAL_BACKOFF_SECONDS * (2 ** (retries - 1))
                logger.warning(
                    f"LLM Throttling para {context_log}. Tentativa {retries}/{AppConstants.MAX_RETRIES}. "
                    f"Aguardando {wait_time:.2f}s. Erro: {e}"
                )
                time.sleep(wait_time)
            except Exception as e:
                resultado["error"] = str(e)
                logger.error(
                    f"Erro na invocação do LLM para {context_log}: {resultado['error']}"
                )
                return resultado
        resultado["error"] = f"Falha no LLM após {AppConstants.MAX_RETRIES} tentativas."
        logger.error(resultado["error"])
        return resultado

    def extrair_dados_de_imagem(
        self, prompt: str, imagem_base64: str, context_log: str
    ) -> Dict:
        """
        Envia uma imagem e um prompt para o LLM e valida a resposta
        com o modelo Pydantic CarteirinhaExtraida.
        """
        corpo = {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 4096,
            "temperature": 0.05,
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image",
                            "source": {
                                "type": "base64",
                                "media_type": "image/jpeg",
                                "data": imagem_base64,
                            },
                        },
                        {"type": "text", "text": prompt},
                    ],
                }
            ],
        }
        resultado = self._invocar_llm(corpo, context_log)
        if resultado["success"]:
            try:
                # Valida o resultado com o modelo da carteirinha
                resultado["dados"] = CarteirinhaExtraida(**resultado["dados"])
            except Exception as pydantic_error:
                resultado.update(
                    {
                        "success": False,
                        "error": f"Erro de validação Pydantic: {pydantic_error}",
                    }
                )
        return resultado

In [41]:
# Teste de extração de dados da carteirinha com o LLM

logger.info("🚀 Testando a extração de dados da carteirinha com o LLM...")
logger.info("=" * 50)

# 1. Configuração do serviço LLM
llm_service = None
try:
    settings = Settings()
    bedrock_client = criar_boto3_client("bedrock-runtime", settings)
    llm_service = LLMService(bedrock_client, settings.BEDROCK_MODEL_ID)
    logger.info("✅ Serviço LLM inicializado com sucesso.")
except Exception as e:
    logger.critical(f"❌ Falha ao inicializar o serviço LLM: {e}")

# 2. Definição do Prompt de Extração
extraction_prompt = """
A imagem fornecida é uma carteirinha de convênio de saúde. Analise a imagem e extraia as seguintes informações em formato JSON:
- convenio: O nome da operadora do plano de saúde.
- plano: O tipo ou nome do plano (ex: "Plano Prata", "Enfermaria").
- nome_pessoa: O nome completo do beneficiário.
- numero_carteirinha: O número de identificação ou matrícula da carteirinha.

Se alguma informação não for encontrada, retorne `null` para o campo correspondente.
O JSON deve ter a seguinte estrutura:
{
  "convenio": "string",
  "plano": "string",
  "nome_pessoa": "string",
  "numero_carteirinha": "string"
}
"""

# 3. Execução da Extração e Coleta de Resultados
extraction_results_list = []
if llm_service and 'base64_images' in locals() and base64_images:
    logger.info(f"🖼️ Encontradas {len(base64_images)} imagens para processar.")
    
    for i, b64_image in enumerate(base64_images):
        context_log = f"carteirinha_imagem_{i+1}"
        logger.info(f"\n--- Processando {context_log} ---")
        
        resultado_extracao = llm_service.extrair_dados_de_imagem(
            prompt=extraction_prompt,
            imagem_base64=b64_image,
            context_log=context_log
        )
        
        if resultado_extracao["success"]:
            dados: CarteirinhaExtraida = resultado_extracao["dados"]
            logger.info("✅ Extração bem-sucedida!")
            logger.info(f"  Convênio: {dados.convenio}")
            logger.info(f"  Plano: {dados.plano}")
            logger.info(f"  Nome: {dados.nome_pessoa}")
            logger.info(f"  Número da Carteirinha: {dados.numero_carteirinha}")
            # Adiciona os dados extraídos à lista
            extraction_results_list.append(dados.model_dump())
        else:
            logger.error(f"❌ Falha na extração para {context_log}: {resultado_extracao['error']}")
            logger.debug(f"Raw response: {resultado_extracao.get('raw_response')}")

elif not llm_service:
    logger.error("❌ O teste não pode ser executado porque o serviço LLM não foi inicializado.")
else:
    logger.warning("⚠️ Nenhuma imagem em base64 foi encontrada para processar. Execute as células anteriores para gerar a variável 'base64_images'.")

# 4. Salvar resultados em um arquivo JSON
if extraction_results_list:
    results_dir = "../documents/extraction_results"
    os.makedirs(results_dir, exist_ok=True)
    results_path = os.path.join(results_dir, "extraction_results.json")
    
    try:
        with open(results_path, 'w', encoding='utf-8') as f:
            json.dump(extraction_results_list, f, ensure_ascii=False, indent=4)
        logger.info(f"✅ {len(extraction_results_list)} resultados da extração salvos com sucesso em: {results_path}")
    except Exception as e:
        logger.error(f"❌ Falha ao salvar o arquivo JSON: {e}")
else:
    logger.warning("⚠️ Nenhum resultado de extração para salvar.")


logger.info("=" * 50)
logger.info("✅ Teste de extração finalizado.")

{"timestamp": "2025-08-06T12:52:46", "level": "INFO", "name": "__main__", "message": "🚀 Testando a extração de dados da carteirinha com o LLM...", "filename": "1154567891.py", "lineno": 3}
{"timestamp": "2025-08-06T12:52:46", "level": "INFO", "name": "__main__", "message": "==================================================", "filename": "1154567891.py", "lineno": 4}
{"timestamp": "2025-08-06T12:52:46", "level": "INFO", "name": "__main__", "message": "==================================================", "filename": "1154567891.py", "lineno": 4}
{"timestamp": "2025-08-06T12:52:46", "level": "INFO", "name": "__main__", "message": "Criando cliente BEDROCK-RUNTIME para a região: us-east-1...", "filename": "788707453.py", "lineno": 5}
{"timestamp": "2025-08-06T12:52:46", "level": "INFO", "name": "__main__", "message": "Cliente BEDROCK-RUNTIME criado com sucesso.", "filename": "788707453.py", "lineno": 15}
{"timestamp": "2025-08-06T12:52:46", "level": "INFO", "name": "__main__", "message": "